In [ ]:
import os
os.chdir(r'C:\Users\bouac\Desktop\Deep-reforcement-learning\deep_project')
print("Répertoire:", os.getcwd())

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import sys
sys.path.append('..')

print("numpy OK:", np.__version__)

In [ ]:
env = SecretEnv0()
print("=== SecretEnv0 ===")
print(f"États   : {env.num_states()}")
print(f"Actions : {env.num_actions()}")
print(f"Rewards : {[env.reward(i) for i in range(env.num_rewards())]}")

In [ ]:
algos = {
    'Monte Carlo ES': lambda: monte_carlo_es(SecretEnv0Adapted(), num_episodes=1000),
    'On-policy MC':   lambda: on_policy_first_visit_mc_control(SecretEnv0Adapted(), num_episodes=1000),
    'Off-policy MC':  lambda: off_policy_mc_control(SecretEnv0Adapted(), num_episodes=1000),
    'Sarsa':          lambda: sarsa(SecretEnv0Adapted(), num_episodes=1000),
    'Q-Learning':     lambda: q_learning(SecretEnv0Adapted(), num_episodes=1000),
}

print("=== Tous les algos sur SecretEnv0 ===\n")
results = {}
for name, algo in algos.items():
    t = time.time()
    pi, Q = algo()
    t = time.time() - t
    q_mean = Q.max(axis=1).mean()
    q_max = Q.max()
    print(f"{name}")
    print(f"  Temps        : {t:.2f}s")
    print(f"  Q max moyen  : {q_mean:.4f}")
    print(f"  Q max global : {q_max:.4f}")
    print()
    results[name] = (pi, Q, t, q_mean)

In [ ]:
names = list(results.keys())
times = [results[n][2] for n in names]
scores = [results[n][3] for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Temps
bars = ax1.bar(names, times, color=['blue', 'green', 'orange', 'red', 'purple'])
ax1.set_ylabel('Temps (s)')
ax1.set_title('Temps exécution — SecretEnv0')
ax1.set_xticklabels(names, rotation=15)
for bar, t in zip(bars, times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{t:.2f}s', ha='center', va='bottom', fontsize=8)

# Q max moyen
bars2 = ax2.bar(names, scores, color=['blue', 'green', 'orange', 'red', 'purple'])
ax2.set_ylabel('Q max moyen')
ax2.set_title('Performance — SecretEnv0')
ax2.set_xticklabels(names, rotation=15)
for bar, s in zip(bars2, scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{s:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Comparaison algos — SecretEnv0', fontsize=14)
plt.tight_layout()
plt.savefig('secret_env0_comparison.png', dpi=100, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé ✅")

In [ ]:
epsilons = [0.01, 0.05, 0.1, 0.2, 0.5]
sarsa_scores = []
ql_scores = []

for eps in epsilons:
    _, Q = sarsa(SecretEnv0Adapted(), num_episodes=5000, epsilon=eps)
    sarsa_scores.append(Q.max(axis=1).mean())

    _, Q = q_learning(SecretEnv0Adapted(), num_episodes=5000, epsilon=eps)
    ql_scores.append(Q.max(axis=1).mean())

    print(f"epsilon={eps} | Sarsa={sarsa_scores[-1]:.4f} | Q-Learning={ql_scores[-1]:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epsilons, sarsa_scores, marker='o', label='Sarsa', color='blue')
ax.plot(epsilons, ql_scores, marker='o', label='Q-Learning', color='orange')
ax.set_xlabel('Epsilon')
ax.set_ylabel('Q max moyen')
ax.set_title('Impact epsilon — SecretEnv0')
ax.legend()
plt.tight_layout()
plt.savefig('secret_env0_epsilon.png', dpi=100, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé ✅")